# Model-personalisation analysis (BioScout)

**Portable template.** This notebook hard-codes no paths. Drop it into any
BioScout project folder (one that has `settings.py`, `models/` and
`simulations/`) and run — `bioscout.Project()` discovers the project root from
the working directory and wires everything up. As long as `bioscout` is
installed in the environment, the notebook runs from wherever it lives.

**Goal.** Quantify how musculoskeletal-model personalisation changes muscle
forces and joint contact forces (JCF), by comparing the models/curves defined
in this project's `settings.py` (`model_config` / `SUBJECTS`):

1. **Generic vs personalised geometry** (e.g. scaled vs MRI)
2. **Static optimisation vs EMG-informed (CEINMS)** — `<model>` vs `<model> - CEINMS`
3. **Across models** — whatever scaled models the project defines

For each contrast we compare: marker (IK) errors, EMG errors, joint angles &
moments, muscle moment arms, muscle moments (force x moment arm), muscle
forces, and joint contact forces (JRA).

---
**How this notebook is organised**
- **0. Setup** — import BioScout, point it at this project, build `model_config`.
- **1. Pipeline (optional re-run)** — `Analyse` runs IK -> ID -> MA -> SO -> JRA,
  EMG normalisation and the CEINMS pipeline per model/trial.
- **2. Comparisons** — one section per metric, looping over the trials.
- **3. Summary** — master results table + markdown report.

Everything is driven by `settings.py` in this project folder. To add/remove a
model curve, edit `SUBJECTS` / `model_config` there — every figure updates.

## 0. Setup — import BioScout and point it at this project

`bioscout.Project()` finds the project root (the folder with `settings.py`),
loads that `settings.py`, and points `utils.MODELS_DIR / SIMULATIONS_DIR /
RESULTS_DIR` at it. We then import `utils` and `settings` as real modules so
the editor (Pylance) gives full autocomplete on both.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import bioscout
proj = bioscout.Project()        # finds project root, loads settings.py, wires utils dirs

from bioscout import utils   # typed package module: utils.Analyse, utils.Plot, utils.Summarize, utils.RESULTS_DIR
import settings              # typed project file: settings.SESSION, settings.trial_list, settings.SUBJECTS, settings.DOFS

# Typed hierarchy: proj.subjects[0].sessions[0].trials[0].run_ik(replace=True)
print(proj, "|", len(proj.subjects), "subjects | session:", settings.SESSION)
print("Subjects:", [s.name for s in proj.subjects])

In [ ]:
# --- Configuration (from the project's subjects) ---------------------------
SESSION       = settings.SESSION
TRIALS        = settings.trial_list
model_config  = bioscout.build_model_config(proj.subjects)
labels        = list(model_config)
colors        = {k: v['color']      for k, v in model_config.items()}
forces_type   = {k: v['force_type'] for k, v in model_config.items()}
lineStyles    = {k: v['line_style'] for k, v in model_config.items()}
subjects      = {k: v['subject']    for k, v in model_config.items()}
DOFS          = settings.DOFS
DOFS_MOMENTS  = settings.DOFS_MOMENTS
MUSCLE_GROUPS = settings.MUSCLE_GROUPS

print(f'{len(labels)} curves over {len(TRIALS)} trials: {TRIALS}')

In [ ]:
# --- Helpers (hierarchy-based: Subject -> Trial) ---------------------------
def load_trials(trial_name, session=None):
    """Return {label: Trial} for one trial. Each Trial IS an Analyse with the
    right model + setup wired in for that label's solver (SO -> increased model,
    CEINMS -> base), via Subject.make_trial. No separate path-repair step needed.
    """
    session = session or settings.SESSION
    out = {}
    for label in labels:
        subj = proj.subject(subjects[label])
        if subj is None:
            print(f"[missing] no subject for {label} ({subjects[label]})"); continue
        out[label] = subj.make_trial(subj.trial_path(trial_name, session),
                                     force_type=forces_type[label])
    return out

def results_dir_for(trial_name):
    d = os.path.join(utils.RESULTS_DIR, "model_comparison", trial_name)
    os.makedirs(d, exist_ok=True)
    return d

## 1. Pipeline — (re-)run or refresh per model and trial

This section runs the full OpenSim + CEINMS pipeline for **every model and
trial** in `model_config`. It is idempotent: BioScout skips a step when its
output already exists **unless** `replace=True`.

Set `RUN_PIPELINE = True` to actually execute (slow; needs OpenSim + CEINMS on
this machine). Leave it `False` to skip straight to the comparisons using the
results already on disk.

The per-trial steps are, in order:
`export_c3d -> run_ik -> run_id -> run_ma -> run_so -> calculate_muscle_moments('so')
-> run_jra` (static-optimisation branch), then the EMG-informed branch
`run_emg_normalise -> CEINMS calibration -> CEINMS execution -> run_jra_ceinms`.

In [ ]:
RUN_PIPELINE = False     # set True to (re)run the OpenSim/CEINMS pipeline
REPLACE      = False     # set True to overwrite existing step outputs

def run_so_branch(trial):
    trial.run_ik(replace=REPLACE); trial.run_id(replace=REPLACE); trial.run_ma(replace=REPLACE)
    trial.run_so(replace=REPLACE)
    trial.calculate_muscle_moments(forces_type='so')
    trial.run_jra(replace=REPLACE)

def run_ceinms_branch(trial):
    trial.run_ik(replace=REPLACE); trial.run_id(replace=REPLACE); trial.run_ma(replace=REPLACE)
    trial.run_emg_normalise()
    trial.create_ceinms_input_data()
    trial.create_ceinms_calibration_cfg()
    trial.create_ceinms_calibration_setup()
    trial.run_ceinms_calibration()
    trial.create_ceinms_exe_cfg()
    trial.create_ceinms_exe_setup()
    trial.run_ceinms_exe()
    trial.calculate_muscle_moments(forces_type='ceinms')
    trial.run_jra_ceinms(replace=REPLACE)

if RUN_PIPELINE:
    for trial_name in TRIALS:
        print(f"\n{'='*70}\nTRIAL: {trial_name}\n{'='*70}")
        for label, trial in load_trials(trial_name).items():
            ft = forces_type[label]
            print(f"\n--- {label} [{ft}] ({trial.path}) ---")
            try:
                run_ceinms_branch(trial) if ft == 'CEINMS' else run_so_branch(trial)
            except Exception as e:
                print(f"[ERROR] {label}/{trial_name}: {e}")
    print("\nPipeline finished.")
else:
    print("RUN_PIPELINE is False -- using existing results on disk.")

## 2. Model comparisons

Each subsection below loops over the trials and compares all models in
`model_config`. Figures are saved to `results/model_comparison/<trial>/` and
also shown inline.

We lean on BioScout's built-in `Plot` class where it already does the job
(`external_biomechanics`, `moment_arms`, `muscle_moments`, `marker_error`) and
add focused cells for the metrics needed in tabular form (marker / EMG /
moment errors) and for joint contact forces.

In [ ]:
# --- 2.0  One Plot object per trial ----------------------------------------
# bioscout.utils.Plot reads settings.model_config and builds its own
# {label: Analyse} dict internally, saving into results_dir.
plots = {}
for trial_name in TRIALS:
    plots[trial_name] = utils.Plot(session=SESSION,
                                   trialName=trial_name,
                                   results_dir=results_dir_for(trial_name))
print("Plot objects ready for:", list(plots.keys()))

### 2.1 Marker (IK) errors

Mean RMS marker error per model — how well each scaled/personalised model
tracks the experimental markers. Lower is better. Built from each trial's
`_ik_marker_errors.sto` via `Analyse.calculate_mean_marker_error()`.

In [ ]:
def marker_error_table(trials):
    rows = []
    for label, trial in trials.items():
        try:
            df = trial.calculate_mean_marker_error()    # single-value DataFrame
            val_m = float(df.iloc[0, 0])
            rows.append({"Model": label, "Marker RMS error (mm)": val_m * 1000.0})
        except Exception as e:
            rows.append({"Model": label, "Marker RMS error (mm)": np.nan})
            print(f"[skip] {label}: {e}")
    return pd.DataFrame(rows).set_index("Model")

marker_tables = {}
for trial_name in TRIALS:
    trials = load_trials(trial_name)
    tbl = marker_error_table(trials)
    marker_tables[trial_name] = tbl
    print(f"\n=== Marker errors — {trial_name} ===")
    display(tbl.round(2))

# Grouped bar across trials
fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(labels))
w = 0.8 / len(TRIALS)
for i, trial_name in enumerate(TRIALS):
    vals = [marker_tables[trial_name]["Marker RMS error (mm)"].get(l, np.nan) for l in labels]
    ax.bar(x + i * w, vals, w, label=trial_name)
ax.set_xticks(x + w * (len(TRIALS) - 1) / 2)
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Marker RMS error (mm)")
ax.set_title("IK marker error by model and trial")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(utils.RESULTS_DIR, "model_comparison", "marker_errors_all_trials.png"), dpi=200)
plt.show()

### 2.2 EMG errors (R2 and RMSE%)

Agreement between measured EMG linear envelopes and modelled muscle
activations, per EMG channel and model. SO activations come from
`SO_StaticOptimization_activation.sto`; CEINMS activations come from the
CEINMS `Execution.../Activations.sto`. For each mapped channel we report R2
and RMSE as a percentage of the EMG range.

In [ ]:
EMG_MAP = settings.CEINMSSettings.emg_muscle_mapping   # channel -> [muscles]

def _emg_envelope(trial):
    """Load the normalised EMG envelope used as CEINMS excitation."""
    for attr in ("emg_filtered_normalised", "ceinms_excitations"):
        p = getattr(trial, attr, None)
        if p:
            fp = p if os.path.isabs(p) else os.path.join(trial.path, p)
            if os.path.exists(fp):
                return utils.load_any_data_file(fp)
    return None

def _activations(trial, kind):
    """kind='so' -> SO activations; kind='ceinms' -> CEINMS activations."""
    if kind == "so":
        fp = os.path.join(trial.path, trial.so_activations)
    else:
        cf = getattr(trial, "jra_forces_ceinms", None)
        if not cf:
            return None
        fp = cf.replace("MuscleForces.sto", "Activations.sto")
        if not os.path.isabs(fp):
            fp = os.path.join(trial.path, fp)
    return utils.load_any_data_file(fp) if os.path.exists(fp) else None

def emg_error_table(trials):
    rows = []
    for label, trial in trials.items():
        kind = "ceinms" if "CEINMS" in label else "so"
        emg = _emg_envelope(trial)
        act = _activations(trial, kind)
        if emg is None or act is None:
            print(f"[skip] {label}: EMG or activation file missing")
            continue
        n = min(len(emg), len(act))
        for channel, muscles in EMG_MAP.items():
            if channel not in emg.columns or not muscles:
                continue
            model_cols = [m for m in muscles if m in act.columns]
            if not model_cols:
                continue
            e = pd.to_numeric(emg[channel], errors="coerce").values[:n]
            a = act[model_cols].mean(axis=1).values[:n]
            rng = np.nanmax(e) - np.nanmin(e)
            rmse_pct = (utils.rmse(e, a) / rng * 100) if rng else np.nan
            rows.append({"Model": label, "EMG channel": channel,
                         "R2": utils.rsquared(e, a), "RMSE %": rmse_pct})
    return pd.DataFrame(rows)

emg_tables = {}
for trial_name in TRIALS:
    trials = load_trials(trial_name)
    df = emg_error_table(trials)
    emg_tables[trial_name] = df
    if not df.empty:
        print(f"\n=== EMG errors — {trial_name} (mean over channels) ===")
        display(df.groupby("Model")[["R2", "RMSE %"]].mean().round(3))

### 2.3 Joint angles and moments

`Plot.external_biomechanics()` overlays inverse-kinematics joint angles (top
row) and inverse-dynamics joint moments (bottom row) for all models, one
column per DOF. Knee angle is sign-flipped for the models listed in
`settings.MODELS_TO_FLIP_KNEE` so every curve shares one convention.

In [ ]:
for trial_name in TRIALS:
    print(f"\n=== Joint angles & moments — {trial_name} ===")
    try:
        plots[trial_name].external_biomechanics()
        plt.show()
    except Exception as e:
        print(f"[error] {trial_name}: {e}")

### 2.4 Moment errors (muscle moments vs inverse dynamics)

For each DOF, the sum of muscle moments (force x moment arm) should reproduce
the inverse-dynamics joint moment. `Analyse.calculate_moment_errors()` returns
RMSE, RMSE% and R2 per DOF — a check on how well each model's SO/CEINMS
solution balances the external moments.

In [ ]:
def moment_error_table(trials):
    out = {}
    for label, trial in trials.items():
        kind = "ceinms" if "CEINMS" in label else "so"
        try:
            me = trial.calculate_moment_errors(forces_type=kind)
            out[label] = me
        except Exception as e:
            print(f"[skip] {label}: {e}")
    return out

moment_error_tables = {}
for trial_name in TRIALS:
    trials = load_trials(trial_name)
    tabs = moment_error_table(trials)
    moment_error_tables[trial_name] = tabs
    if tabs:
        # Mean R2 / RMSE% across the DOFs of interest, per model
        summary = pd.DataFrame({
            lbl: {
                "Moment R2 (mean)":    pd.to_numeric(me.loc[me.index.intersection(DOFS), "R2"], errors="coerce").mean(),
                "Moment RMSE% (mean)": pd.to_numeric(me.loc[me.index.intersection(DOFS), "RMSE %"], errors="coerce").mean(),
            }
            for lbl, me in tabs.items()
        }).T
        print(f"\n=== Moment errors — {trial_name} ===")
        display(summary.round(3))

### 2.5 Muscle moment arms

`Plot.moment_arms()` draws, for every DOF, one subplot per muscle with a
non-zero moment arm, overlaying all models. This exposes how bone-geometry
personalisation (MRI) and model choice change the muscles' leverage. Figures
are written to `results/model_comparison/<trial>/moment_arms_<dof>.png`.

In [ ]:
for trial_name in TRIALS:
    print(f"\n=== Moment arms — {trial_name} ===")
    try:
        plots[trial_name].moment_arms()
    except Exception as e:
        print(f"[error] {trial_name}: {e}")

### 2.6 Muscle moments (force x moment arm)

`Plot.muscle_moments()` plots each muscle's contribution to the net joint
moment per DOF, comparing models (and SO vs CEINMS where configured).

In [ ]:
for trial_name in TRIALS:
    print(f"\n=== Muscle moments — {trial_name} ===")
    try:
        plots[trial_name].muscle_moments()
        plt.show()
    except Exception as e:
        print(f"[error] {trial_name}: {e}")

### 2.7 Muscle forces by group

Summed muscle force per functional group (gluteals, adductors, hamstrings,
quadriceps, triceps surae...) for each model. SO uses
`SO_StaticOptimization_force.sto`; CEINMS uses the Execution
`MuscleForces.sto`. This is the primary outcome for the *generic vs MRI* and
*SO vs CEINMS* contrasts.

In [ ]:
def _force_df(trial, kind):
    if kind == "so":
        fp = os.path.join(trial.path, trial.so_forces)
    else:
        cf = getattr(trial, "jra_forces_ceinms", None)
        fp = cf if (cf and os.path.isabs(cf)) else (os.path.join(trial.path, cf) if cf else None)
    return utils.load_any_data_file(fp) if (fp and os.path.exists(fp)) else None

def plot_muscle_forces(trial_name):
    trials = load_trials(trial_name)
    n_cols = 5
    n_rows = int(np.ceil(len(MUSCLE_GROUPS) / n_cols))
    fig, ax = plt.subplots(n_rows, n_cols, figsize=(4.2 * n_cols, 2.6 * n_rows), sharex=True)
    ax = ax.flatten()
    for count, (group, muscles) in enumerate(MUSCLE_GROUPS.items()):
        ax[count].set_title(group, fontsize=9)
        for label, trial in trials.items():
            kind = "ceinms" if "CEINMS" in label else "so"
            df = _force_df(trial, kind)
            if df is None:
                continue
            cols = [m for m in muscles if m in df.columns]
            if not cols:
                continue
            ax[count].plot(df["time"], df[cols].sum(axis=1), label=label,
                           color=colors[label], linestyle=lineStyles[label], linewidth=1.3)
    for i in range(len(MUSCLE_GROUPS), len(ax)):
        ax[i].axis("off")
    h, l = ax[0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=4, frameon=False, fontsize=8)
    fig.suptitle(f"Muscle forces by group — {trial_name}", fontsize=12)
    fig.subplots_adjust(left=0.05, right=0.98, top=0.92, bottom=0.13, wspace=0.2, hspace=0.3)
    save = os.path.join(results_dir_for(trial_name), f"muscle_forces_{trial_name}.png")
    plt.savefig(save, dpi=200, bbox_inches="tight")
    plt.show()
    print("saved:", save)

for trial_name in TRIALS:
    print(f"\n=== Muscle forces — {trial_name} ===")
    try:
        plot_muscle_forces(trial_name)
    except Exception as e:
        print(f"[error] {trial_name}: {e}")

### 2.8 Joint contact forces (JRA)

Hip, knee and ankle contact forces from the Joint Reaction Analysis. The knee
joint is defined differently per model, so column names are resolved through
`settings.JRA_COLUMNS(model)`. SO-based JCF is in
`Analyse_JRA_ReactionLoads_SO.sto`; CEINMS-based JCF in
`Analyse_JRA_ReactionLoads_CEINMS.sto`. Columns shown: X (ant/post), Y
(med/lat), Z (sup/inf) and the resultant magnitude. This is the key outcome
for both hypotheses.

In [ ]:
JOINT_ROW = {"hip": 0, "knee": 1, "ankle": 2}

def plot_jcf(trial_name):
    trials = load_trials(trial_name)
    fig, ax = plt.subplots(3, 4, figsize=(15, 8), sharex=True)
    for label, trial in trials.items():
        kind = "ceinms" if "CEINMS" in label else "so"
        jra_file = trial.jra_ceinms if kind == "ceinms" else trial.jra
        fp = os.path.join(trial.path, jra_file)
        if not os.path.exists(fp):
            print(f"[skip] {label}: {jra_file} not found")
            continue
        loads = utils.load_any_data_file(fp)
        comps = settings.JRA_COLUMNS(label)
        for joint, cols in comps.items():
            missing = [c for c in cols if c not in loads.columns]
            if missing:
                print(f"[skip] {label}/{joint}: missing {missing[:1]}...")
                continue
            r = JOINT_ROW[joint]
            x, y, z = loads[cols[0]], loads[cols[1]], loads[cols[2]]
            res = np.linalg.norm([x, y, z], axis=0)
            for c, series in zip(range(3), (x, y, z)):
                ax[r, c].plot(loads["time"], series, label=label,
                              color=colors[label], linestyle=lineStyles[label])
            ax[r, 3].plot(loads["time"], res, label=label,
                          color=colors[label], linestyle=lineStyles[label])
            ax[r, 0].set_ylabel(f"{joint.capitalize()} force (N)")
    ax[0, 0].set_title("X (+ant / -post)")
    ax[0, 1].set_title("Y (+med / -lat)")
    ax[0, 2].set_title("Z (+sup / -inf)")
    ax[0, 3].set_title("Resultant magnitude")
    for c in range(4):
        ax[2, c].set_xlabel("Time (s)")
    h, l = ax[0, 0].get_legend_handles_labels()
    uniq = dict(zip(l, h))
    fig.legend(uniq.values(), uniq.keys(), loc="lower center", ncol=4, fontsize=8, frameon=False)
    fig.suptitle(f"Joint contact forces — {trial_name}", fontsize=12)
    fig.subplots_adjust(bottom=0.12, top=0.93, hspace=0.15)
    save = os.path.join(results_dir_for(trial_name), f"joint_contact_forces_{trial_name}.png")
    plt.savefig(save, dpi=200, bbox_inches="tight")
    plt.show()
    print("saved:", save)

def jcf_peak_table(trial_name):
    """Peak resultant JCF per joint and model — compact numeric comparison."""
    trials = load_trials(trial_name)
    rows = []
    for label, trial in trials.items():
        kind = "ceinms" if "CEINMS" in label else "so"
        jra_file = trial.jra_ceinms if kind == "ceinms" else trial.jra
        fp = os.path.join(trial.path, jra_file)
        if not os.path.exists(fp):
            continue
        loads = utils.load_any_data_file(fp)
        comps = settings.JRA_COLUMNS(label)
        row = {"Model": label}
        for joint, cols in comps.items():
            if all(c in loads.columns for c in cols):
                res = np.linalg.norm([loads[cols[0]], loads[cols[1]], loads[cols[2]]], axis=0)
                row[f"{joint} peak (N)"] = float(np.nanmax(res))
                row[f"{joint} peak (BW)"] = float(np.nanmax(res)) / (trial.body_mass * 9.81) \
                    if getattr(trial, "body_mass", None) else np.nan
        rows.append(row)
    return pd.DataFrame(rows).set_index("Model")

jcf_peaks = {}
for trial_name in TRIALS:
    print(f"\n=== Joint contact forces — {trial_name} ===")
    try:
        plot_jcf(trial_name)
        jcf_peaks[trial_name] = jcf_peak_table(trial_name)
        display(jcf_peaks[trial_name].round(1))
    except Exception as e:
        print(f"[error] {trial_name}: {e}")

## 3. Summary across contrasts

A compact read of the hypotheses, plus a machine-readable master table.

- **Generic vs MRI** — defined in `settings.CONTRASTS["generic_vs_mri"]`
- **SO vs CEINMS** — each model's `-` vs `- CEINMS`
- **Across models** — the scaled models in the project

`Summarize` walks the simulations tree and writes a long-format
`master_summary.csv` (all IK/ID/SO/CEINMS time series, normalised to 0-100%),
ready for stats or external plotting.

In [ ]:
# --- 3.1  Peak JCF focused on the contrasts --------------------------------
def contrast_view(metric_tables, value_col=None):
    """Stack per-trial tables and tag each row with the trial it belongs to."""
    frames = []
    for trial_name, tbl in metric_tables.items():
        if tbl is None or (hasattr(tbl, "empty") and tbl.empty):
            continue
        t = tbl.copy()
        t["trial"] = trial_name
        frames.append(t.reset_index())
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

jcf_all = contrast_view(jcf_peaks)
if not jcf_all.empty:
    print("Peak joint contact forces — all trials & models:")
    display(jcf_all.round(1))

    # Generic vs MRI — knee & hip resultant peaks
    contrast = getattr(settings, "CONTRASTS", {}).get("generic_vs_mri", [])
    if contrast:
        gen_mri = jcf_all[jcf_all["Model"].isin(contrast)]
        print("\nGeneric vs MRI — peak resultant JCF:")
        display(gen_mri.set_index(["trial", "Model"]).filter(like="peak (N)").round(1))

In [ ]:
# --- 3.2  Master long-format results table ---------------------------------
BUILD_MASTER = False        # set True to (re)build results/master_summary.csv
if BUILD_MASTER:
    s = utils.Summarize()
    master = s.create_master_df("master_summary.csv")
    print("master_summary.csv shape:", None if master is None else master.shape)
else:
    print("BUILD_MASTER is False — skipping master table build.")

In [ ]:
# --- 3.3  Markdown report of the generated figures -------------------------
for trial_name in TRIALS:
    try:
        path = plots[trial_name].write_results_markdown_summary("summary.md")
        print("Wrote:", path)
    except Exception as e:
        print(f"[error] {trial_name}: {e}")

---
### Notes & next steps

- **Portability:** this notebook hard-codes no paths. Copy it into any BioScout
  project folder and run; `bioscout.Project()` resolves the root from the cwd.
- **Re-running:** set `RUN_PIPELINE = True` (S1) to regenerate every model's
  OpenSim/CEINMS outputs, and `REPLACE = True` to overwrite existing files.
- **Adding/removing a curve:** edit `SUBJECTS` / `model_config` in `settings.py`;
  all figures and tables update automatically.
- **CEINMS forces** are read from each trial's `Execution_a{a}_b{b}_g{g}/`
  folder via the `jra_forces_ceinms` attribute restored from
  `trial_settings.xml`. If a CEINMS curve is missing, run S1 for that
  model/trial first.